In [ ]:
# Colab/bootstrap installs
!pip -q install xgboost weightwatcher  scikit-learn pandas matplotlib seaborn scipy
!pip -q install git+https://github.com/CalculatedContent/xgboost2ww.git


<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/SpamBase_Hyperparameter_Sweep_Alpha2_ManifoldSearch_W1278910.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpamBase hyperparameter sweep and manifold search (multi-W)

This notebook now treats `W` as an explicit experimental axis and evaluates each trained model under every matrix family in `W_LIST`.


## Why the notebook is now multi-W

- A single-W workflow can bias search toward one representation family.
- This notebook keeps model training shared but performs structural diagnostics separately for each matrix kind.
- Sweep diagnostics, anchors, refinement, and winners are all computed per W and compared side by side.


## What stays shared across W

- Train/validation/test split.
- Trained model for each `(hyperparameters, seed, stage)` run.
- Train/validation/test accuracy metrics.
- Hyperparameter configuration definitions.

## What changes with W

- Converted matrix/layer created by `convert(..., W=...)`.
- WeightWatcher alpha diagnostics.
- Sweep helpfulness classification.
- Low-alpha competitive anchors.
- Per-W frontiers and final winners.


## 1. Setup and imports


## 1b. Google Drive checkpointing (Colab)

This mounts Drive in Colab and writes caches/results to Drive so long sweeps can be resumed after disconnects.

In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CHECKPOINT_ROOT_COLAB = Path('/content/drive/MyDrive/xgboost2ww_checkpoints')
CHECKPOINT_ROOT_LOCAL = Path('./checkpoints')
RESULTS_ROOT_COLAB = Path('/content/drive/MyDrive/xgboost2ww_results')
RESULTS_ROOT_LOCAL = Path('./results')
EXPERIMENT_NAME = 'spambase_alpha2_manifoldsearch_w1278910'

if IN_COLAB:
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_COLAB
    RESULTS_ROOT = RESULTS_ROOT_COLAB
else:
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_LOCAL
    RESULTS_ROOT = RESULTS_ROOT_LOCAL

CHECKPOINT_DIR = CHECKPOINT_ROOT / EXPERIMENT_NAME
RESULTS_DIR = RESULTS_ROOT / EXPERIMENT_NAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import hashlib
import json
import pickle
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import xgboost as xgb
import weightwatcher as ww

from tqdm.auto import tqdm

from xgboost2ww import convert
try:
    from xgboost2ww import compute_w10, to_linear_layer
except Exception:
    compute_w10, to_linear_layer = None, None


In [ ]:
RNG = 123
np.random.seed(RNG)

W_LIST = ["W1", "W2", "W7", "W8", "W9", "W10"]
TARGET_ALPHA = 2.0
HARD_ALPHA_BAND = (1.85, 2.15)
SOFT_ALPHA_BAND = (1.70, 2.30)
VAL_COMPETITIVE_TOL = 0.002
N_COARSE_SEEDS = 3
N_REFINE_SEEDS = 5
ENABLE_OPTIONAL_SECONDARY_DIAGNOSTICS = False

TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
WW_ANALYZE_OPTIONS = dict(randomize=True, detX=True, fix_fingers='clip_xmax')

CACHE_DIR = CHECKPOINT_DIR / "cache_multiw"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Dataset loading and fixed split


In [ ]:
def load_spambase_data():
    try:
        ds = fetch_openml(name='spambase', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float32)
        y = np.asarray(ds.target)
        if y.dtype.kind in 'OUS':
            y = np.where(np.isin(y.astype(str), ['1', 'spam', 'yes', 'True']), 1, 0)
        y = y.astype(int)
        source = 'openml'
    except Exception:
        df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data', header=None)
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
        y = df.iloc[:, -1].to_numpy(dtype=int)
        source = 'uci_csv'
    if np.isnan(X).any():
        med = np.nanmedian(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = med[inds[1]]
    return X, y, source

X, y, data_source = load_spambase_data()
idx_train_full, idx_test = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y)
idx_train, idx_valid = train_test_split(idx_train_full, test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=y[idx_train_full])

X_train, y_train = X[idx_train], y[idx_train]
X_valid, y_valid = X[idx_valid], y[idx_valid]
X_test, y_test = X[idx_test], y[idx_test]
X_train_valid, y_train_valid = X[idx_train_full], y[idx_train_full]

print('data_source=', data_source, 'train/valid/test=', X_train.shape, X_valid.shape, X_test.shape)


## 3. Baseline model and common coarse sweep definition


In [ ]:
BASE_OBJECTIVE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc'],
    'tree_method': 'hist'
}

BASELINE_PARAMS = dict(
    max_depth=5,
    eta=0.08,
    n_estimators=1600,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
)


def unique_sorted(values):
    return sorted({float(v) for v in values})


def build_coarse_sweep_configs(base):
    families = {}
    families['n_estimators'] = unique_sorted([base['n_estimators']*f for f in [0.65,0.8,1.0,1.25,1.45]])
    families['max_depth'] = sorted({max(2, int(base['max_depth']+d)) for d in [-2,-1,0,1,2]})
    families['min_child_weight'] = unique_sorted([max(1.0, base['min_child_weight']*f) for f in [0.5,0.75,1.0,1.5,2.0]])
    families['subsample'] = unique_sorted([min(1.0,max(0.5, base['subsample']+d)) for d in [-0.2,-0.1,0.0,0.1,0.15]])
    families['eta_x_n_estimators'] = [(base['eta']*f, int(base['n_estimators']/f)) for f in [0.8,0.9,1.0,1.1,1.2]]

    configs = []
    configs.append(dict(sweep_name='baseline', sweep_value='baseline', params=deepcopy(base)))
    for key, vals in families.items():
        for v in vals:
            p = deepcopy(base)
            if key == 'eta_x_n_estimators':
                p['eta'] = float(v[0]); p['n_estimators'] = int(v[1])
                sval = f"eta={p['eta']:.5f}|n_estimators={p['n_estimators']}"
            elif key == 'max_depth':
                p[key] = int(v); sval = str(int(v))
            elif key == 'n_estimators':
                p[key] = int(v); sval = str(int(v))
            else:
                p[key] = float(v); sval = f"{float(v):.6g}"
            configs.append(dict(sweep_name=key, sweep_value=sval, params=p))

    dedup = {}
    for c in configs:
        sig = json.dumps(c['params'], sort_keys=True)
        if sig not in dedup:
            dedup[sig] = c
    return list(dedup.values())

coarse_configs = build_coarse_sweep_configs(BASELINE_PARAMS)
print('coarse configurations:', len(coarse_configs))


## 4. Shared model-training cache


In [ ]:
TRAINING_CACHE = {}


def canonicalize_params(params):
    return json.dumps(params, sort_keys=True, separators=(',', ':'))


def make_configuration_id(params, stage_name=None):
    payload = {'params': json.loads(canonicalize_params(params)), 'stage_name': stage_name or 'na'}
    digest = hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:12]
    return f"cfg_{digest}"


def train_model_once(params, seed):
    train_params = dict(BASE_OBJECTIVE_PARAMS)
    train_params.update({k: v for k, v in params.items() if k != 'n_estimators'})
    train_params['seed'] = int(seed)

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtest = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(
        train_params,
        dtrain,
        num_boost_round=int(params['n_estimators']),
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )
    best_round = int(getattr(model, 'best_iteration', params['n_estimators'] - 1) + 1)

    pred_train = (model.predict(dtrain, iteration_range=(0, best_round)) > 0.5).astype(int)
    pred_valid = (model.predict(dvalid, iteration_range=(0, best_round)) > 0.5).astype(int)
    pred_test = (model.predict(dtest, iteration_range=(0, best_round)) > 0.5).astype(int)

    metrics = dict(
        train_accuracy=float(accuracy_score(y_train, pred_train)),
        val_accuracy=float(accuracy_score(y_valid, pred_valid)),
        test_accuracy=float(accuracy_score(y_test, pred_test)),
        best_round=best_round,
    )
    return model, metrics, train_params


def train_or_load_model(params, seed, stage_name, sweep_name='na', sweep_value='na'):
    cfg_id = make_configuration_id(params, stage_name=stage_name)
    key = f"{cfg_id}|seed={seed}"
    cache_path = CACHE_DIR / f"{key}.pkl"

    if key in TRAINING_CACHE:
        return TRAINING_CACHE[key]
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            run = pickle.load(f)
        TRAINING_CACHE[key] = run
        return run

    model, metrics, train_params = train_model_once(params, seed)
    run = {
        'configuration_id': cfg_id,
        'stage_name': stage_name,
        'seed': int(seed),
        'sweep_name': sweep_name,
        'sweep_value': sweep_value,
        'params': deepcopy(params),
        'model': model,
        'metrics': metrics,
        'train_params': train_params,
        'best_round': int(metrics['best_round']),
        'cached_model_key': key,
    }
    with open(cache_path, 'wb') as f:
        pickle.dump(run, f)
    TRAINING_CACHE[key] = run
    return run


## 5. Multi-W WeightWatcher evaluation helpers


In [ ]:
def extract_weightwatcher_metrics(details_df, matrix_kind):
    out = {
        'matrix_kind': matrix_kind,
        'alpha_primary': np.nan,
        'alpha_mean': np.nan,
        'alpha_min': np.nan,
        'n_valid_layers': 0,
        'rand_num_spikes': np.nan,
        'matrix_rows': np.nan,
        'matrix_cols': np.nan,
        'matrix_rank': np.nan,
    }
    if details_df is None or len(details_df) == 0:
        return out

    dd = details_df.copy()
    if 'alpha' in dd.columns:
        alpha = pd.to_numeric(dd['alpha'], errors='coerce').dropna()
        if len(alpha):
            out['alpha_primary'] = float(alpha.iloc[0])
            out['alpha_mean'] = float(alpha.mean())
            out['alpha_min'] = float(alpha.min())
            out['n_valid_layers'] = int(alpha.shape[0])
    for c in ['rand_num_spikes', 'M', 'N', 'matrix_rank']:
        if c in dd.columns and dd[c].notna().any():
            val = pd.to_numeric(dd[c], errors='coerce').dropna()
            if len(val):
                if c == 'M': out['matrix_rows'] = float(val.iloc[0])
                elif c == 'N': out['matrix_cols'] = float(val.iloc[0])
                else: out[c] = float(val.iloc[0])
    return out


def _convert_layer_for_w(model, train_params, best_round, W):
    last_err = None
    for return_type in ['torch', 'numpy']:
        try:
            layer = convert(
                model=model,
                data=X_train_valid,
                labels=y_train_valid,
                W=W,
                nfolds=5,
                t_points=40,
                random_state=RNG,
                train_params=train_params,
                num_boost_round=best_round,
                multiclass='error',
                return_type=return_type,
                verbose=False,
            )
            return layer
        except Exception as e:
            last_err = e
    if W == 'W10' and compute_w10 is not None and to_linear_layer is not None:
        try:
            W10_mat = compute_w10(model, X_train_valid, y_train_valid)
            return to_linear_layer(W10_mat)
        except Exception as e:
            last_err = e
    raise last_err


def evaluate_all_ws_for_trained_model(trained_run, W_LIST=W_LIST):
    rows = []
    for W in W_LIST:
        row = {
            'configuration_id': trained_run['configuration_id'],
            'stage_name': trained_run['stage_name'],
            'seed': trained_run['seed'],
            'sweep_name': trained_run['sweep_name'],
            'sweep_value': trained_run['sweep_value'],
            **trained_run['params'],
            'train_accuracy': trained_run['metrics']['train_accuracy'],
            'val_accuracy': trained_run['metrics']['val_accuracy'],
            'test_accuracy': trained_run['metrics']['test_accuracy'],
            'matrix_kind': W,
            'status': 'ok',
            'error_msg': None,
        }
        try:
            layer = _convert_layer_for_w(trained_run['model'], trained_run['train_params'], trained_run['best_round'], W)
            details_df = ww.WeightWatcher(model=layer).analyze(**WW_ANALYZE_OPTIONS)
            row.update(extract_weightwatcher_metrics(details_df, W))
        except Exception as e:
            row.update(dict(alpha_primary=np.nan, alpha_mean=np.nan, alpha_min=np.nan, n_valid_layers=0,
                            rand_num_spikes=np.nan, matrix_rows=np.nan, matrix_cols=np.nan, matrix_rank=np.nan))
            row['status'] = 'error'
            row['error_msg'] = f"{type(e).__name__}: {e}"
        rows.append(row)
    return rows


## 6. Stage A: common coarse sweep, train once / analyze all W


In [ ]:
coarse_trained_runs = []
coarse_ww_rows = []
coarse_seeds = [RNG + i for i in range(N_COARSE_SEEDS)]

for cfg in tqdm(coarse_configs, desc='Stage A coarse configs'):
    for seed in coarse_seeds:
        run = train_or_load_model(cfg['params'], seed=seed, stage_name='coarse', sweep_name=cfg['sweep_name'], sweep_value=cfg['sweep_value'])
        coarse_trained_runs.append(run)
        coarse_ww_rows.extend(evaluate_all_ws_for_trained_model(run, W_LIST=W_LIST))

trained_runs_df = pd.DataFrame([
    {
        'configuration_id': r['configuration_id'], 'stage_name': r['stage_name'], 'seed': r['seed'],
        'sweep_name': r['sweep_name'], 'sweep_value': r['sweep_value'], **r['params'],
        'train_accuracy': r['metrics']['train_accuracy'], 'val_accuracy': r['metrics']['val_accuracy'],
        'test_accuracy': r['metrics']['test_accuracy'], 'best_round': r['best_round'],
        'cached_model_key': r['cached_model_key'],
    }
    for r in coarse_trained_runs
])
ww_results_long_df = pd.DataFrame(coarse_ww_rows)
print(trained_runs_df.shape, ww_results_long_df.shape)


In [ ]:
def aggregate_tables(trained_runs_df, ww_results_long_df):
    metric_aggs = dict(train_accuracy=['mean','std'], val_accuracy=['mean','std'], test_accuracy=['mean','std'], best_round=['mean'])
    aggregated_models_df = trained_runs_df.groupby('configuration_id', as_index=False).agg(metric_aggs)
    aggregated_models_df.columns = ['_'.join(c).strip('_') for c in aggregated_models_df.columns.to_flat_index()]

    meta_cols = ['configuration_id','stage_name','sweep_name','sweep_value','max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']
    meta = trained_runs_df[meta_cols].drop_duplicates('configuration_id')
    aggregated_models_df = aggregated_models_df.merge(meta, on='configuration_id', how='left')

    agg = ww_results_long_df.groupby(['configuration_id','matrix_kind'], as_index=False).agg({
        'train_accuracy':'mean','val_accuracy':'mean','test_accuracy':'mean',
        'alpha_primary':'mean','alpha_mean':'mean','alpha_min':'mean',
        'n_valid_layers':'mean','rand_num_spikes':'mean','matrix_rows':'mean','matrix_cols':'mean','matrix_rank':'mean'
    })
    agg = agg.rename(columns={
        'train_accuracy':'train_accuracy_mean','val_accuracy':'val_accuracy_mean','test_accuracy':'test_accuracy_mean',
        'alpha_primary':'alpha_primary_mean','alpha_mean':'alpha_mean_mean','alpha_min':'alpha_min_mean',
    })
    agg = agg.merge(meta, on='configuration_id', how='left')
    agg['alpha_distance_to_2_mean'] = (agg['alpha_primary_mean'] - TARGET_ALPHA).abs()
    agg['train_val_gap_mean'] = agg['train_accuracy_mean'] - agg['val_accuracy_mean']
    agg['train_test_gap_mean'] = agg['train_accuracy_mean'] - agg['test_accuracy_mean']
    return aggregated_models_df, agg

aggregated_models_df, aggregated_multiw_df = aggregate_tables(trained_runs_df, ww_results_long_df)
aggregated_multiw_df.head(3)


## 7. Stage B: per-W sweep diagnostics and manifold classification


In [ ]:
def _safe_spearman(x, y):
    d = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(d) < 3 or d['x'].nunique() < 2 or d['y'].nunique() < 2:
        return np.nan
    return float(spearmanr(d['x'], d['y']).correlation)


def _safe_slope(x, y):
    d = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(d) < 3 or d['x'].nunique() < 2:
        return np.nan
    return float(np.polyfit(d['x'], d['y'], 1)[0])


def compute_sweep_diagnostics_per_w(aggregated_multiw_df):
    rows = []
    grp = aggregated_multiw_df.groupby(['matrix_kind','sweep_name'], dropna=False)
    for (w,s), g in grp:
        rho_val = _safe_spearman(g['alpha_primary_mean'], g['val_accuracy_mean'])
        rho_test = _safe_spearman(g['alpha_primary_mean'], g['test_accuracy_mean'])
        rho_gap = _safe_spearman(g['alpha_primary_mean'], g['train_test_gap_mean'])
        slope_val = _safe_slope(g['alpha_primary_mean'], g['val_accuracy_mean'])
        slope_test = _safe_slope(g['alpha_primary_mean'], g['test_accuracy_mean'])
        if pd.notna(rho_test) and rho_test > 0.15 and pd.notna(slope_test) and slope_test > 0:
            cls = 'alpha_helpful'
        elif pd.notna(rho_test) and rho_test < -0.15 and pd.notna(slope_test) and slope_test < 0:
            cls = 'capacity_tradeoff'
        else:
            cls = 'inconclusive'
        rows.append(dict(
            matrix_kind=w, sweep_name=s, n_configs=len(g),
            alpha_min=float(g['alpha_primary_mean'].min()), alpha_max=float(g['alpha_primary_mean'].max()),
            val_min=float(g['val_accuracy_mean'].min()), val_max=float(g['val_accuracy_mean'].max()),
            test_min=float(g['test_accuracy_mean'].min()), test_max=float(g['test_accuracy_mean'].max()),
            spearman_alpha_val=rho_val, spearman_alpha_test=rho_test, spearman_alpha_train_test_gap=rho_gap,
            slope_val_vs_alpha=slope_val, slope_test_vs_alpha=slope_test,
            sweep_classification=cls,
        ))
    return pd.DataFrame(rows)

sweep_diagnostics_df = compute_sweep_diagnostics_per_w(aggregated_multiw_df)
display(sweep_diagnostics_df.sort_values(['matrix_kind','sweep_name']).head(20))


In [ ]:
heat_test = sweep_diagnostics_df.pivot(index='matrix_kind', columns='sweep_name', values='spearman_alpha_test')
plt.figure(figsize=(10,4))
sns.heatmap(heat_test, annot=True, cmap='coolwarm', center=0)
plt.title('Spearman(alpha, test_acc) by matrix_kind x sweep_name')
plt.tight_layout(); plt.show()


## 8. Stage C: per-W anchor selection


In [ ]:
ACCURACY_BASELINE = aggregated_models_df.sort_values('val_accuracy_mean', ascending=False).iloc[0]
LOW_ALPHA_COMPETITIVE_ANCHOR_BY_W = {}
anchor_rows = []

for W in W_LIST:
    wdf = aggregated_multiw_df[aggregated_multiw_df['matrix_kind'] == W].copy()
    d_w = sweep_diagnostics_df[sweep_diagnostics_df['matrix_kind'] == W]
    helpful = d_w.loc[d_w['sweep_classification']=='alpha_helpful', 'sweep_name'].tolist()
    cand = wdf[wdf['sweep_name'].isin(helpful)].copy() if helpful else wdf.copy()
    if len(cand) == 0:
        cand = wdf.copy()
    if len(cand) == 0:
        continue
    best_val_w = float(cand['val_accuracy_mean'].max())
    comp = cand[cand['val_accuracy_mean'] >= best_val_w - VAL_COMPETITIVE_TOL].copy()
    if len(comp) == 0:
        comp = cand.copy()
    anchor = comp.sort_values(['alpha_primary_mean','val_accuracy_mean'], ascending=[True,False]).iloc[0]
    LOW_ALPHA_COMPETITIVE_ANCHOR_BY_W[W] = anchor
    anchor_rows.append(anchor)

anchors_by_w_df = pd.DataFrame(anchor_rows)
display(anchors_by_w_df[['matrix_kind','configuration_id','sweep_name','alpha_primary_mean','val_accuracy_mean','test_accuracy_mean']])


## 9. Stage D: per-W targeted refinement proposal


In [ ]:
def propose_refinement_configs_for_w(anchor_row, diagnostics_for_w):
    anchor = {k: anchor_row[k] for k in ['max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']}
    helpful = set(diagnostics_for_w.loc[diagnostics_for_w['sweep_classification']=='alpha_helpful','sweep_name'])
    tradeoff = set(diagnostics_for_w.loc[diagnostics_for_w['sweep_classification']=='capacity_tradeoff','sweep_name'])

    props = []
    def add(p):
        q = deepcopy(anchor); q.update(p)
        q['max_depth'] = int(max(2, round(q['max_depth'])))
        q['n_estimators'] = int(max(200, round(q['n_estimators'])))
        q['subsample'] = float(min(1.0, max(0.5, q['subsample'])))
        q['min_child_weight'] = float(max(1.0, q['min_child_weight']))
        props.append(q)

    add({})
    if 'subsample' in helpful:
        for d in [-0.08,-0.04,0.04,0.08]: add({'subsample': anchor['subsample']+d})
    if 'max_depth' in tradeoff:
        for d in [-1,0,1]: add({'max_depth': anchor['max_depth']+d})
    elif 'max_depth' in helpful:
        for d in [-2,-1,1]: add({'max_depth': anchor['max_depth']+d})
    if 'min_child_weight' in helpful:
        for f in [0.75,0.9,1.1,1.35]: add({'min_child_weight': anchor['min_child_weight']*f})
    if 'eta_x_n_estimators' in helpful:
        for f in [0.9,0.95,1.05,1.1]: add({'eta': anchor['eta']*f, 'n_estimators': anchor['n_estimators']/f})
    for s in diagnostics_for_w.loc[diagnostics_for_w['sweep_classification']=='inconclusive', 'sweep_name']:
        if s == 'subsample':
            add({'subsample': anchor['subsample']+0.03})
        if s == 'min_child_weight':
            add({'min_child_weight': anchor['min_child_weight']*1.15})

    dedup = {}
    for p in props:
        dedup[canonicalize_params(p)] = p
    return list(dedup.values())

proposals_by_w = {}
ranked_union = []
for W in W_LIST:
    if W not in LOW_ALPHA_COMPETITIVE_ANCHOR_BY_W:
        continue
    anchor = LOW_ALPHA_COMPETITIVE_ANCHOR_BY_W[W]
    dw = sweep_diagnostics_df[sweep_diagnostics_df['matrix_kind']==W]
    proposals = propose_refinement_configs_for_w(anchor, dw)
    gap_ref = float(anchor['train_val_gap_mean']) if 'train_val_gap_mean' in anchor else 0.01
    for p in proposals:
        score = float(anchor['val_accuracy_mean'] - 0.003*abs(anchor['alpha_primary_mean']-TARGET_ALPHA) - 0.20*max(0.0, anchor['train_val_gap_mean']-gap_ref))
        ranked_union.append((W, score, p))
    proposals_by_w[W] = proposals

refine_candidates = []
for W in proposals_by_w:
    topk = sorted([r for r in ranked_union if r[0]==W], key=lambda x: x[1], reverse=True)[:10]
    refine_candidates.extend([r[2] for r in topk])

dedup = {}
for p in refine_candidates:
    dedup[canonicalize_params(p)] = p
refine_configs = list(dedup.values())
print('refinement unique configs:', len(refine_configs))


## 10. Stage E: train union of refined configurations once / analyze all W


In [ ]:
refined_trained_runs = []
refined_ww_rows = []
refine_seeds = [RNG + 100 + i for i in range(N_REFINE_SEEDS)]

for p in tqdm(refine_configs, desc='Stage E refined configs'):
    for seed in refine_seeds:
        run = train_or_load_model(p, seed=seed, stage_name='refine', sweep_name='refine_union', sweep_value='multiw')
        refined_trained_runs.append(run)
        refined_ww_rows.extend(evaluate_all_ws_for_trained_model(run, W_LIST=W_LIST))

if refined_trained_runs:
    add_runs = pd.DataFrame([
        {'configuration_id': r['configuration_id'], 'stage_name': r['stage_name'], 'seed': r['seed'], 'sweep_name': r['sweep_name'],
         'sweep_value': r['sweep_value'], **r['params'], 'train_accuracy': r['metrics']['train_accuracy'],
         'val_accuracy': r['metrics']['val_accuracy'], 'test_accuracy': r['metrics']['test_accuracy'],
         'best_round': r['best_round'], 'cached_model_key': r['cached_model_key']}
        for r in refined_trained_runs
    ])
    trained_runs_df = pd.concat([trained_runs_df, add_runs], ignore_index=True)
    ww_results_long_df = pd.concat([ww_results_long_df, pd.DataFrame(refined_ww_rows)], ignore_index=True)

aggregated_models_df, aggregated_multiw_df = aggregate_tables(trained_runs_df, ww_results_long_df)
sweep_diagnostics_df = compute_sweep_diagnostics_per_w(aggregated_multiw_df)


## 11. Stage F: per-W Pareto/frontier analysis


In [ ]:
def pareto_frontier_max_y(df, x_col, y_col):
    d = df[[x_col, y_col, 'configuration_id']].dropna().sort_values(x_col, ascending=True).copy()
    keep, best = [], -np.inf
    for _, r in d.iterrows():
        if r[y_col] > best:
            keep.append(r['configuration_id']); best = r[y_col]
    return df[df['configuration_id'].isin(keep)].copy()


def compute_frontiers_for_w(aggregated_multiw_df, matrix_kind, val_tol=VAL_COMPETITIVE_TOL):
    wdf = aggregated_multiw_df[aggregated_multiw_df['matrix_kind']==matrix_kind].copy()
    if len(wdf) == 0:
        return []
    best_val_row = wdf.sort_values('val_accuracy_mean', ascending=False).iloc[0]
    best_val = float(best_val_row['val_accuracy_mean'])

    comp = wdf[wdf['val_accuracy_mean'] >= best_val - val_tol].copy()
    low_alpha = comp.sort_values(['alpha_primary_mean','val_accuracy_mean'], ascending=[True,False]).iloc[0]

    pareto = pareto_frontier_max_y(wdf, 'alpha_primary_mean', 'val_accuracy_mean')
    pareto_row = pareto.assign(score=(pareto['val_accuracy_mean'] - 0.002*(pareto['alpha_primary_mean']-TARGET_ALPHA).abs())).sort_values('score', ascending=False).iloc[0]

    hard = wdf[wdf['alpha_primary_mean'].between(HARD_ALPHA_BAND[0], HARD_ALPHA_BAND[1])]
    if len(hard):
        alpha2 = hard.sort_values('val_accuracy_mean', ascending=False).iloc[0]
    else:
        soft = wdf[wdf['alpha_primary_mean'].between(SOFT_ALPHA_BAND[0], SOFT_ALPHA_BAND[1])]
        if len(soft):
            alpha2 = soft.sort_values('val_accuracy_mean', ascending=False).iloc[0]
        else:
            alpha2 = comp.sort_values(['alpha_distance_to_2_mean','val_accuracy_mean'], ascending=[True,False]).iloc[0]

    out = []
    for typ, row in [('best_val_accuracy_model_by_w',best_val_row),('best_low_alpha_model_by_w',low_alpha),('best_pareto_model_by_w',pareto_row),('best_alpha2_model_by_w',alpha2)]:
        d = row.to_dict(); d['selection_type']=typ; d['matrix_kind']=matrix_kind; out.append(d)
    return out

final_rows = []
for W in W_LIST:
    final_rows.extend(compute_frontiers_for_w(aggregated_multiw_df, W))
final_models_by_w_df = pd.DataFrame(final_rows)
display(final_models_by_w_df[['matrix_kind','selection_type','configuration_id','val_accuracy_mean','test_accuracy_mean','alpha_primary_mean']])


## 12. Stage G: per-W final shortlist and cross-W comparison


In [ ]:
for W in W_LIST:
    wdf = aggregated_multiw_df[aggregated_multiw_df['matrix_kind']==W]
    if len(wdf)==0:
        continue
    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_primary_mean', y='val_accuracy_mean', hue='stage_name', alpha=0.7)
    plt.title(f'{W}: validation accuracy vs alpha_primary_mean'); plt.tight_layout(); plt.show()

    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_primary_mean', y='test_accuracy_mean', hue='stage_name', alpha=0.7)
    plt.title(f'{W}: test accuracy vs alpha_primary_mean'); plt.tight_layout(); plt.show()

    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_distance_to_2_mean', y='val_accuracy_mean', hue='stage_name', alpha=0.7)
    plt.title(f'{W}: validation accuracy vs alpha_distance_to_2_mean'); plt.tight_layout(); plt.show()

    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_distance_to_2_mean', y='test_accuracy_mean', hue='stage_name', alpha=0.7)
    plt.title(f'{W}: test accuracy vs alpha_distance_to_2_mean'); plt.tight_layout(); plt.show()

    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_primary_mean', y='train_test_gap_mean', hue='stage_name', alpha=0.7)
    plt.title(f'{W}: train_test_gap_mean vs alpha_primary_mean'); plt.tight_layout(); plt.show()

    p1 = pareto_frontier_max_y(wdf, 'alpha_primary_mean', 'val_accuracy_mean')
    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_primary_mean', y='val_accuracy_mean', alpha=0.2)
    sns.lineplot(data=p1.sort_values('alpha_primary_mean'), x='alpha_primary_mean', y='val_accuracy_mean', color='red')
    plt.title(f'{W}: Pareto frontier alpha_primary_mean vs val_accuracy_mean'); plt.tight_layout(); plt.show()

    p2 = pareto_frontier_max_y(wdf, 'alpha_distance_to_2_mean', 'val_accuracy_mean')
    plt.figure(figsize=(5,4)); sns.scatterplot(data=wdf, x='alpha_distance_to_2_mean', y='val_accuracy_mean', alpha=0.2)
    sns.lineplot(data=p2.sort_values('alpha_distance_to_2_mean'), x='alpha_distance_to_2_mean', y='val_accuracy_mean', color='purple')
    plt.title(f'{W}: Pareto frontier alpha_distance_to_2_mean vs val_accuracy_mean'); plt.tight_layout(); plt.show()

heat_val = sweep_diagnostics_df.pivot(index='matrix_kind', columns='sweep_name', values='spearman_alpha_val')
heat_test = sweep_diagnostics_df.pivot(index='matrix_kind', columns='sweep_name', values='spearman_alpha_test')
plt.figure(figsize=(10,4)); sns.heatmap(heat_val, annot=True, cmap='coolwarm', center=0); plt.title('Spearman(alpha, val_acc)'); plt.tight_layout(); plt.show()
plt.figure(figsize=(10,4)); sns.heatmap(heat_test, annot=True, cmap='coolwarm', center=0); plt.title('Spearman(alpha, test_acc)'); plt.tight_layout(); plt.show()

bar_alpha = final_models_by_w_df[final_models_by_w_df['selection_type']=='best_low_alpha_model_by_w'][['matrix_kind','alpha_primary_mean']]
plt.figure(figsize=(8,3)); sns.barplot(data=bar_alpha, x='matrix_kind', y='alpha_primary_mean'); plt.axhline(TARGET_ALPHA, ls='--', c='r'); plt.title('Best competitive alpha by matrix'); plt.tight_layout(); plt.show()

bar_val = final_models_by_w_df[final_models_by_w_df['selection_type']=='best_val_accuracy_model_by_w'][['matrix_kind','val_accuracy_mean']]
plt.figure(figsize=(8,3)); sns.barplot(data=bar_val, x='matrix_kind', y='val_accuracy_mean'); plt.title('Best validation accuracy by matrix'); plt.tight_layout(); plt.show()

for wa, wb in [('W7','W8'),('W8','W9'),('W8','W10')]:
    pa = aggregated_multiw_df[aggregated_multiw_df['matrix_kind']==wa][['configuration_id','alpha_primary_mean']].rename(columns={'alpha_primary_mean':f'alpha_{wa}'})
    pb = aggregated_multiw_df[aggregated_multiw_df['matrix_kind']==wb][['configuration_id','alpha_primary_mean']].rename(columns={'alpha_primary_mean':f'alpha_{wb}'})
    m = pa.merge(pb, on='configuration_id', how='inner')
    if len(m)==0: continue
    plt.figure(figsize=(4,4)); sns.scatterplot(data=m, x=f'alpha_{wa}', y=f'alpha_{wb}', alpha=0.6)
    plt.title(f'Alpha comparison: {wa} vs {wb}'); plt.tight_layout(); plt.show()

display(final_models_by_w_df.sort_values(['matrix_kind','selection_type']))


## 13. Save CSVs and plots


In [ ]:
trained_runs_df.to_csv(RESULTS_DIR / 'spambase_trained_runs.csv', index=False)
ww_results_long_df.to_csv(RESULTS_DIR / 'spambase_ww_results_long_allW.csv', index=False)
aggregated_models_df.to_csv(RESULTS_DIR / 'spambase_aggregated_models.csv', index=False)
aggregated_multiw_df.to_csv(RESULTS_DIR / 'spambase_aggregated_multiw.csv', index=False)
sweep_diagnostics_df.to_csv(RESULTS_DIR / 'spambase_sweep_diagnostics_allW.csv', index=False)
anchors_by_w_df.to_csv(RESULTS_DIR / 'spambase_anchors_by_w.csv', index=False)
final_models_by_w_df.to_csv(RESULTS_DIR / 'spambase_final_models_by_w.csv', index=False)

print('Saved all required multi-W result tables to', RESULTS_DIR)
